## Setup and Installation

### Pass 1 Dependencies
Run this block if `RUN_MODELS` includes GBDTs or `sap_rpt`.

In [2]:
# !pip install git+https://github.com/SAP-samples/sap-rpt-1-oss.git
# !pip install xgboost lightgbm catboost optuna huggingface_hub kaggle

### Pass 2 Dependencies
Run this block **only** if `RUN_MODELS = ['tabfm']` after a runtime restart.

In [ ]:
!git clone https://github.com/google-research/tabfm.git
%cd tabfm
!pip install -e .[pytorch]
%cd ..

import os
os.kill(os.getpid(), 9)

fatal: destination path 'tabfm' already exists and is not an empty directory.
/content/tabfm
Obtaining file:///content/tabfm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for tabfm (pyproject.toml) ... done
  Created wheel for tabfm: filename=tabfm-1.0.1-py3-none-any.whl size=7315 sha256=105f9472c42b7b036a8f7e79863a3fa7d4fd1d75df2bd108812f39caa0de66a4
  Stored in directory: /tmp/pip-ephem-wheel-cache-9ftmryc2/wheels/66/03/52/c5f66697de19c1b27e228605a8223989bd6d85508bc3f65f33
Successfully built tabfm
  Attempting uninstall: tabfm
    Found existing installation: tabfm 1.0.1
    Uninstalling tabfm-1.0.1:
      Successfully uninstalled tabfm-1.0.1


## Data Acquisition

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import hashlib
from typing import Tuple
import optuna
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                             mean_squared_error, mean_absolute_error, r2_score)

# Securely load foundation models
try:
    from sap_rpt_oss import SAP_RPT_OSS_Classifier, SAP_RPT_OSS_Regressor
except ImportError:
    print('sap_rpt_oss not installed yet.')

try:
    from tabfm import TabFMClassifier, TabFMRegressor
    from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0
    # Pre-load the base model to save time in the loop
    tabfm_base = tabfm_v1_0_0.load()
except ImportError:
    print('tabfm not installed yet.')

In [11]:
import os
import pandas as pd
from google.colab import files

file_path = 'WA_Fn-UseC_-HR-Employee-Attrition.csv'

if not os.path.exists(file_path):
    if not os.path.exists('ibm-hr-analytics-attrition-dataset.zip'):
        print("Please upload your kaggle.json file to download the dataset.")
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            !kaggle datasets download -d pavansubhasht/ibm-hr-analytics-attrition-dataset

    if os.path.exists('ibm-hr-analytics-attrition-dataset.zip'):
        !unzip -o ibm-hr-analytics-attrition-dataset.zip

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    df = df.drop(columns=['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber'])
    print("Dataset loaded successfully into 'df'.")
else:
    print("Error: Dataset not found. Please upload the CSV manually or provide kaggle.json.")

Dataset loaded successfully into 'df'.


## Global Configuration
Define which models to run in this pass. Remember to restart the runtime when switching between `sap_rpt` and `tabfm` due to dependency conflicts.

In [12]:
# import getpass
# from huggingface_hub import login

# login(token=getpass.getpass("HF token:"))

In [13]:
# from huggingface_hub import whoami, hf_hub_download
# print(whoami()["name"])
# print(hf_hub_download("SAP/sap-rpt-1-oss", "2025-11-04_sap-rpt-one-oss.pt"))

In [14]:
# This cell is now a placeholder. TabFM logic has been integrated into the main evaluation loop below.

In [15]:
import numpy as np
import pandas as pd
import hashlib
from typing import Tuple

# PASS 1: ["xgboost", "lightgbm", "catboost", "sap_rpt"]
# PASS 2: ["tabfm"]
RUN_MODELS = ["tabfm"]
RANDOM_SEED = 42

def mask_semantics(X: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    """
    Renames columns to col_00, col_01... and replaces categorical strings
    with deterministic opaque tokens while keeping numeric values byte-identical.
    """
    X_masked = X.copy()
    np.random.seed(seed)

    # 1. Mask Column Names
    new_cols = [f"col_{i:02d}" for i in range(len(X.columns))]
    rename_map = dict(zip(X.columns, new_cols))
    X_masked.rename(columns=rename_map, inplace=True)

    # 2. Mask Non-Numeric Values
    for col in X_masked.columns:
        if not pd.api.types.is_numeric_dtype(X_masked[col]):
            unique_levels = sorted(X_masked[col].unique().astype(str))
            permuted_indices = np.random.permutation(len(unique_levels))

            level_map = {}
            for i, level in enumerate(unique_levels):
                idx = permuted_indices[i]
                token = f"cat_{hashlib.md5(str(idx).encode()).hexdigest()[:4]}"
                level_map[level] = token

            X_masked[col] = X_masked[col].astype(str).map(level_map)

    return X_masked

print("Robust masking function initialized.")

Robust masking function initialized.


## Task Setup
We prepare Task A (Binary Classification: Attrition) and Task B (Regression: MonthlyIncome).

In [16]:
# Task A: Classification Preparation
if 'df' in globals():
    y_a = df['Attrition'].map({'Yes': 1, 'No': 0})
    X_a = df.drop(columns=['Attrition'])

    # Task B: Regression Preparation
    # JobLevel is dropped because it correlates ~0.95 with MonthlyIncome, making it trivial.
    y_b = df['MonthlyIncome']
    X_b = df.drop(columns=['Attrition', 'MonthlyIncome', 'JobLevel'])

    print(f"Task A features: {X_a.shape[1]}, Task B features: {X_b.shape[1]}")
else:
    print("DataFrame 'df' not found. Please run the Data Acquisition cell (9f2b0c48) first.")

Task A features: 30, Task B features: 28


## Model Implementation and Evaluation Loop

This section defines the wrappers for GBDTs (with Optuna tuning) and the foundation models. It then executes the full experimental grid.

In [17]:
import time
import numpy as np
import pandas as pd
import optuna
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                             mean_squared_error, mean_absolute_error, r2_score)
from sap_rpt_oss import SAP_RPT_OSS_Classifier, SAP_RPT_OSS_Regressor

optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = 3

def get_gbdt_model(name, task, trial_or_params):
    is_trial = isinstance(trial_or_params, (optuna.trial.Trial, optuna.trial.FrozenTrial))
    def get_p(pname, low, high, log=False):
        if is_trial: return trial_or_params.suggest_int(pname, low, high) if isinstance(low, int) else trial_or_params.suggest_float(pname, low, high, log=log)
        return trial_or_params[pname]

    if name == "xgboost":
        params = {"n_estimators": get_p("n_estimators", 100, 1000), "learning_rate": trial_or_params.suggest_float("learning_rate", 0.001, 0.3, log=True) if is_trial else trial_or_params["learning_rate"], "max_depth": get_p("max_depth", 3, 10), "random_state": 42, "n_jobs": -1}
        return XGBClassifier(**params) if task == "clf" else XGBRegressor(**params)
    elif name == "lightgbm":
        params = {"n_estimators": get_p("n_estimators", 100, 1000), "learning_rate": trial_or_params.suggest_float("learning_rate", 0.001, 0.3, log=True) if is_trial else trial_or_params["learning_rate"], "num_leaves": get_p("num_leaves", 20, 150), "random_state": 42, "verbosity": -1}
        return LGBMClassifier(**params) if task == "clf" else LGBMRegressor(**params)
    elif name == "catboost":
        params = {"iterations": get_p("iterations", 100, 1000), "learning_rate": trial_or_params.suggest_float("learning_rate", 0.001, 0.3, log=True) if is_trial else trial_or_params["learning_rate"], "depth": get_p("depth", 4, 10), "random_seed": 42, "verbose": False}
        return CatBoostClassifier(**params) if task == "clf" else CatBoostRegressor(**params)

def encode_for_gbdt(X_tr, X_te):
    X_tr, X_te = X_tr.copy(), X_te.copy()
    for c in X_tr.columns:
        if not pd.api.types.is_numeric_dtype(X_tr[c]):
            le = {v: i for i, v in enumerate(sorted(X_tr[c].astype(str).unique()))}
            X_tr[c] = X_tr[c].astype(str).map(le).fillna(-1).astype(np.int64)
            X_te[c] = X_te[c].astype(str).map(le).fillna(-1).astype(np.int64)
    return X_tr, X_te

def fit_sap(task, X_tr, y_tr, X_te, y_te):
    cls = SAP_RPT_OSS_Classifier if task == "clf" else SAP_RPT_OSS_Regressor
    model = cls(max_context_size=2048, bagging=2)
    t0 = time.time()
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    preds = model.predict(X_te)
    if task == "clf":
        probs = model.predict_proba(X_te)[:, 1]
        res = {"auc": roc_auc_score(y_te, probs), "f1_macro": f1_score(y_te, preds, average='macro')}
    else:
        res = {"r2": r2_score(y_te, preds), "rmse": np.sqrt(mean_squared_error(y_te, preds))}
    res["inference_time_s"] = elapsed
    return res

def fit_gbdt(name, task, X_tr, y_tr, X_te, y_te):
    X_tr_enc, X_te_enc = encode_for_gbdt(X_tr, X_te)
    metric = "roc_auc" if task == "clf" else "neg_root_mean_squared_error"
    def objective(trial):
        m = get_gbdt_model(name, task, trial)
        return cross_val_score(m, X_tr_enc, y_tr, cv=3, scoring=metric).mean()
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    best_model = get_gbdt_model(name, task, study.best_params)
    t0 = time.time()
    best_model.fit(X_tr_enc, y_tr)
    elapsed = time.time() - t0
    preds = best_model.predict(X_te_enc)
    if task == "clf":
        probs = best_model.predict_proba(X_te_enc)[:, 1]
        res = {"auc": roc_auc_score(y_te, probs), "f1_macro": f1_score(y_te, preds, average='macro')}
    else:
        res = {"r2": r2_score(y_te, preds), "rmse": np.sqrt(mean_squared_error(y_te, preds))}
    res["train_time_s"] = elapsed
    return res

def run_experiment():
    results = []
    tasks = [("TaskA", X_a, y_a, "clf"), ("TaskB", X_b, y_b, "reg")]
    for task_name, X_base, y_base, task_type in tasks:
        for var_name in ["semantic", "masked"]:
            X = (X_base if var_name == "semantic" else mask_semantics(X_base)).reset_index(drop=True)
            y = y_base.reset_index(drop=True)
            fold_gen = StratifiedKFold(5, shuffle=True, random_state=42) if task_type == "clf" else KFold(5, shuffle=True, random_state=42)
            for fold_idx, (tr_idx, te_idx) in enumerate(fold_gen.split(X, y if task_type == "clf" else None)):
                X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
                y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
                for model_name in RUN_MODELS:
                    try:
                        if model_name in ["xgboost", "lightgbm", "catboost"]:
                            res = fit_gbdt(model_name, task_type, X_tr, y_tr, X_te, y_te)
                        elif model_name == "sap_rpt":
                            res = fit_sap(task_type, X_tr, y_tr, X_te, y_te)
                        else: continue
                        res.update({"task": task_name, "variant": var_name, "fold": fold_idx, "model": model_name})
                        results.append(res)
                        key = "auc" if task_type == "clf" else "r2"
                        print(f"{task_name} | {var_name} | Fold {fold_idx} | {model_name} score: {res.get(key, 0):.4f}")
                    except Exception as e:
                        print(f"Error {model_name} on {task_name}/{var_name}: {e}")
    final_df = pd.DataFrame(results)
    final_df.to_csv("raw_pass1.csv", index=False)
    return final_df

results_df = run_experiment()

In [18]:
Xm = mask_semantics(X_a)
print("Verification - Masked Columns:", Xm.columns[:5].tolist())
print("Verification - Masked Values:")
display(Xm.select_dtypes(exclude='number').head(3).iloc[:, :3])

Verification - Masked Columns: ['col_00', 'col_01', 'col_02', 'col_03', 'col_04']
Verification - Masked Values:


,col_01,col_03,col_06
0,cat_c81e,cat_cfcd,cat_c4ca
1,cat_c4ca,cat_c81e,cat_c4ca
2,cat_c81e,cat_c81e,cat_a87f


In [ ]:
# Ensure configuration and base model are available
try:
    RUN_MODELS
except NameError:
    RUN_MODELS = ['tabfm']

if "tabfm" in RUN_MODELS:
    print("Running TabFM Quick Demo...")
    # Using X_a and y_a which are defined in the cells above
    try:
        m = TabFMClassifier(model=tabfm_base)
        m.fit(X_a.iloc[:800], y_a[:800])
        probs = m.predict_proba(X_a.iloc[800:830])
        print("First 3 predictions:", probs[:3])
    except NameError as e:
        print(f"Initialization error: {e}. Please ensure you have run the data preparation cells (Task A/B).")

Running TabFM Quick Demo...
